In [ ]:
import numpy as np
import plotly.graph_objects as go

# 全局图对象，用于连续添加椭球
_current_fig = None

def ellipsoid(
    a=1,
    b=1,
    c=1,
    center=(0, 0, 0),
    rotation=None,
    u_res=80,
    v_res=80,
    u_range=(0, 2*np.pi),
    v_range=(0, np.pi),
    color="rgba(100,150,255,0.7)",
    wireframe=False,
    name="Ellipsoid",
    show_axes=False,
    fig=None,
    auto_add=True,
    **kwargs,
):
    """
    绘制椭球。

    参数：
        a, b, c: 椭球三个方向的半轴长度（默认 1 为球体）
        center: 椭球中心 (x, y, z)
        rotation: 旋转角度 (rx, ry, rz)（欧拉角，度数），None 表示无旋转
        u_res, v_res: 网格分辨率（越大越精细，默认 30）
        u_range: u 参数范围 (u_min, u_max)，默认 (0, 2π) 为完整圆周
        v_range: v 参数范围 (v_min, v_max)，默认 (0, π) 为完整球面
        color: 椭球颜色（支持 rgba、hex、named color）
        wireframe: True 时绘制网格线而非实心
        name: 图例名称
        show_axes: True 时显示坐标轴
        fig: 指定 Figure 对象，None 时使用全局图或创建新的
        auto_add: 如果 True 且 fig=None，自动使用全局图（连续添加）；
                 如果 False，创建新图（默认 True）
        **kwargs: 其他 go.Surface 参数（如 hovertemplate）

    返回：
        fig (go.Figure)
    
    常用范围示例：
        半球: v_range=(0, np.pi/2)
        四分之一球: u_range=(0, np.pi/2), v_range=(0, np.pi/2)
        球顶部: v_range=(0, np.pi/4)
        半周圆: u_range=(0, np.pi), v_range=(0, np.pi)
    
    使用示例（连续添加到同一图）:
        fig = start_figure()  # 启动新图
        ellipsoid(a=1, b=1, c=1, color='red', name='Sphere 1')
        ellipsoid(a=2, b=1.5, c=1, center=(3,0,0), color='blue', name='Sphere 2')
        show_figure(fig)  # 显示组合图
    """
    global _current_fig
    
    # 参数化椭球面
    u = np.linspace(u_range[0], u_range[1], u_res)
    v = np.linspace(v_range[0], v_range[1], v_res)
    u_grid, v_grid = np.meshgrid(u, v)

    # 基础椭球坐标
    x = a * np.cos(u_grid) * np.sin(v_grid)
    y = b * np.sin(u_grid) * np.sin(v_grid)
    z = c * np.cos(v_grid)

    # 旋转处理（欧拉角 Z-Y-X 约定）
    if rotation:
        rx, ry, rz = (
            np.radians(rotation[0]),
            np.radians(rotation[1]),
            np.radians(rotation[2]),
        )
        # 绕 X 轴旋转
        Rx = np.array(
            [[1, 0, 0], [0, np.cos(rx), -np.sin(rx)], [0, np.sin(rx), np.cos(rx)]]
        )
        # 绕 Y 轴旋转
        Ry = np.array(
            [[np.cos(ry), 0, np.sin(ry)], [0, 1, 0], [-np.sin(ry), 0, np.cos(ry)]]
        )
        # 绕 Z 轴旋转
        Rz = np.array(
            [[np.cos(rz), -np.sin(rz), 0], [np.sin(rz), np.cos(rz), 0], [0, 0, 1]]
        )
        R = Rz @ Ry @ Rx  # 组合旋转矩阵

        # 应用旋转（需要展平、旋转、再reshape）
        pts = np.stack([x, y, z], axis=-1)
        shape = pts.shape
        pts_flat = pts.reshape(-1, 3)
        pts_rot = pts_flat @ R.T
        pts_rotated = pts_rot.reshape(shape)
        x, y, z = pts_rotated[..., 0], pts_rotated[..., 1], pts_rotated[..., 2]

    # 平移到中心
    x = x + center[0]
    y = y + center[1]
    z = z + center[2]

    # 确定要使用的 Figure
    if fig is None:
        if auto_add and _current_fig is not None:
            fig = _current_fig  # 使用全局图
        else:
            fig = go.Figure()  # 创建新图

    # 添加 Surface
    if wireframe:
        # 网格模式：只显示线条
        for i in range(v_res):
            fig.add_trace(
                go.Scatter3d(
                    x=x[i],
                    y=y[i],
                    z=z[i],
                    mode="lines",
                    line=dict(color=color, width=1),
                    showlegend=(i == 0),
                    name=name,
                    hoverinfo="skip",
                )
            )
        for j in range(u_res):
            fig.add_trace(
                go.Scatter3d(
                    x=x[:, j],
                    y=y[:, j],
                    z=z[:, j],
                    mode="lines",
                    line=dict(color=color, width=1),
                    showlegend=False,
                    hoverinfo="skip",
                )
            )
    else:
        # 实心模式：使用 color 参数创建均匀颜色
        surface_kwargs = {k: v for k, v in kwargs.items() if k not in ['u_range', 'v_range']}
        
        # 创建均匀的 surfacecolor（所有点使用相同颜色）
        surfacecolor = np.ones_like(z)
        
        fig.add_trace(
            go.Surface(
                x=x,
                y=y,
                z=z,
                surfacecolor=surfacecolor,
                colorscale=[[0, color], [1, color]],
                showscale=False,
                name=name,
                **surface_kwargs,
            )
        )

    # 设置坐标轴
    axis_dict = dict(showgrid=True, zeroline=show_axes)
    fig.update_layout(
        scene=dict(
            xaxis=axis_dict, yaxis=axis_dict, zaxis=axis_dict, aspectmode="data"
        ),
        hovermode="closest",
    )

    return fig


def start_figure(title="Ellipsoids", width=900, height=700):
    """
    启动一个新的全局图，用于连续添加椭球。
    
    参数：
        title: 图表标题
        width: 图表宽度
        height: 图表高度
    
    返回：
        fig (go.Figure)
    
    使用示例：
        fig = start_figure("My Ellipsoids")
        ellipsoid(a=1, name='Red')
        ellipsoid(a=2, center=(3,0,0), name='Blue')
        show_figure(fig)
    """
    global _current_fig
    _current_fig = go.Figure()
    _current_fig.update_layout(
        title=title,
        width=width,
        height=height,
        showlegend=True,
        scene=dict(aspectmode='data')
    )
    return _current_fig


def show_figure(fig=None):
    """
    显示当前图或指定的图。
    
    参数：
        fig: 要显示的 Figure，None 时显示全局图
    """
    global _current_fig
    if fig is None:
        fig = _current_fig
    if fig is not None:
        fig.show()
    else:
        print("No figure to show. Use start_figure() first.")


def clear_figure():
    """清空全局图。"""
    global _current_fig
    _current_fig = None


def ellipsoids(*ellipsoid_args, title="Ellipsoids", figsize=(900, 700)):
    """
    一次性绘制多个椭球。

    参数：
        *ellipsoid_args: 可变数量的椭球参数字典，每个字典传给 ellipsoid()
        title: 图标题
        figsize: (width, height)

    示例：
        fig = ellipsoids(
            {'a': 1, 'b': 2, 'c': 0.5, 'color': 'red'},
            {'a': 1.5, 'b': 1, 'c': 2, 'center': (2, 0, 0), 'color': 'blue'}
        )
    """
    fig = go.Figure()
    for args in ellipsoid_args:
        ellipsoid(**args, fig=fig)

    fig.update_layout(title=title, width=figsize[0], height=figsize[1], showlegend=True)
    return fig

# 演示基础用法
# fig1 = ellipsoid(name="Sphere", auto_add=False)
fig1 = ellipsoid(a=4,b=1,c=1,u_range=(0,np.pi),v_range=(0,np.pi), color='rgba(255,100,100,0.7)', name="Half Ellipsoid", auto_add=False)
fig1.show()

In [ ]:
# ========== u,v 范围控制演示 ==========
# u: 周向角度参数，范围 [0, 2π]
# v: 极向角度参数，范围 [0, π]

# 示例 1: 完整椭球（默认）
fig_full = ellipsoid(a=2, b=1.5, c=1, name='Full Ellipsoid')
fig_full.show()

# 示例 2: 半球（v 范围 0 到 π/2）
fig_half = ellipsoid(
    a=2, b=1.5, c=1,
    v_range=(0, np.pi/2),
    name='Half Ellipsoid',
    color='rgba(255, 100, 100, 0.7)'
)
fig_half.show()

# 示例 3: 四分之一椭球（u 和 v 都是 π/2）
fig_quarter = ellipsoid(
    a=2, b=1.5, c=1,
    u_range=(0, np.pi/2),
    v_range=(0, np.pi/2),
    name='Quarter Ellipsoid',
    color='rgba(100, 255, 100, 0.7)'
)
fig_quarter.show()

# 示例 4: 八分之一椭球（更小的范围）
fig_eighth = ellipsoid(
    a=2, b=1.5, c=1,
    u_range=(0, np.pi/4),
    v_range=(0, np.pi/2),
    name='Eighth Ellipsoid',
    color='rgba(100, 100, 255, 0.7)'
)
fig_eighth.show()

# 示例 5: 椭球顶部（小的 v 范围）
fig_top = ellipsoid(
    a=2, b=2, c=1,
    v_range=(0, np.pi/6),
    u_range=(0, 2*np.pi),
    name='Ellipsoid Top',
    color='rgba(255, 255, 100, 0.7)',
    u_res=60, v_res=20
)
fig_top.show()

# 示例 6: 多个切片组合
fig_slices = ellipsoid(
    a=1.5, b=1, c=1.2,
    u_range=(0, np.pi/2),
    v_range=(0, np.pi),
    name='Slice 1',
    color='rgba(255, 0, 0, 0.6)'
)
ellipsoid(
    a=1.5, b=1, c=1.2,
    u_range=(np.pi/2, np.pi),
    v_range=(0, np.pi),
    name='Slice 2',
    color='rgba(0, 255, 0, 0.6)',
    fig=fig_slices
)
fig_slices.show()

# 示例 7: 旋转 + 范围控制
fig_rot_range = ellipsoid(
    a=2, b=1, c=0.5,
    u_range=(0, np.pi/2),
    v_range=(0, np.pi/2),
    rotation=(45, 30, 60),
    name='Rotated Quarter',
    color='rgba(200, 150, 255, 0.7)'
)
fig_rot_range.show()

# 示例 8: 网格模式 + 范围控制
fig_wire_range = ellipsoid(
    a=2, b=1.5, c=1,
    u_range=(0, np.pi),
    v_range=(0, np.pi),
    wireframe=True,
    color='rgba(0, 0, 0, 0.8)',
    name='Semi-wireframe',
    u_res=40, v_res=40
)
fig_wire_range.show()

In [ ]:
# ========== 连续添加椭球到同一图 ==========
print("方式 1: 使用全局图连续添加")

# 启动一个新图
fig = start_figure("组合椭球 - 方式1", width=1000, height=800)

# 连续添加多个椭球到同一图
ellipsoid(a=1, b=1, c=1, color='rgba(255, 0, 0, 0.7)', name='Red Sphere')
ellipsoid(a=2, b=1.5, c=0.8, center=(4, 0, 0), color='rgba(0, 255, 0, 0.7)', name='Green Ellipsoid')
ellipsoid(a=1.5, b=1, c=2, center=(2, 3, 0), color='rgba(0, 0, 255, 0.7)', name='Blue Ellipsoid')
ellipsoid(a=0.8, b=0.8, c=1.2, center=(6, 3, 1), rotation=(45, 30, 60), color='rgba(255, 255, 0, 0.7)', name='Rotated Yellow')

# 显示组合图
show_figure(fig)

print("\n方式 2: 显式传递 fig 参数")

# 创建一个新图
fig2 = go.Figure()
fig2.update_layout(title="组合椭球 - 方式2", width=1000, height=800, showlegend=True, scene=dict(aspectmode='data'))

# 明确地把 fig2 传给每个调用
ellipsoid(a=1.2, b=1.2, c=1.2, color='rgba(255, 100, 100, 0.6)', name='Pink Sphere', fig=fig2)
ellipsoid(a=2, b=1, c=0.5, center=(3, 0, 0), color='rgba(100, 100, 255, 0.6)', name='Flat Blue', fig=fig2)
ellipsoid(a=1, b=2, c=1, center=(1.5, 3, 0), color='rgba(100, 255, 100, 0.6)', name='Tall Green', fig=fig2)

fig2.show()

print("\n方式 3: 混合使用不同的部分范围")

clear_figure()  # 清空之前的全局图
fig3 = start_figure("复杂组合 - 半球+切片", width=1200, height=800)

# 添加完整椭球
ellipsoid(a=1, b=1, c=1, center=(0, 0, 0), color='rgba(255, 0, 0, 0.5)', name='Full Ellipsoid')

# 添加半球
ellipsoid(a=1.5, b=1.5, c=1.5, center=(3.5, 0, 0), v_range=(0, np.pi/2), color='rgba(0, 255, 0, 0.6)', name='Hemisphere')

# 添加四分之一球
ellipsoid(a=1.2, b=1.2, c=1.2, center=(0, 3, 0), u_range=(0, np.pi/2), v_range=(0, np.pi/2), color='rgba(0, 0, 255, 0.6)', name='Quarter')

# 添加旋转的四分之一球
ellipsoid(a=2, b=1, c=0.5, center=(3.5, 3, 0), u_range=(0, np.pi/2), v_range=(0, np.pi/2), rotation=(45, 45, 0), color='rgba(255, 255, 0, 0.6)', name='Rotated Quarter')

show_figure(fig3)

print("\n方式 4: 构建复杂结构（多层次）")

clear_figure()
fig4 = start_figure("多层椭球结构", width=1200, height=800)

# 底层 - 大的半透明球体
ellipsoid(a=3, b=3, c=2, center=(0, 0, 0), color='rgba(200, 150, 255, 0.2)', name='Base Layer', u_res=40, v_res=40)

# 中层 - 三个小球
for i in range(3):
    angle = i * 2 * np.pi / 3
    x_pos = 2 * np.cos(angle)
    y_pos = 2 * np.sin(angle)
    ellipsoid(
        a=0.8, b=0.8, c=0.8,
        center=(x_pos, y_pos, 0.5),
        color=f'rgba({100+i*50}, {150-i*30}, {200}, 0.8)',
        name=f'Mid Sphere {i+1}'
    )

# 顶层 - 小的尖锐椭球
ellipsoid(a=1, b=0.5, c=1.5, center=(0, 0, 2.5), color='rgba(255, 200, 100, 0.9)', name='Top Ellipsoid')

show_figure(fig4)

In [ ]:
fig = start_figure("组合椭球 - 方式1", width=1000, height=800)

# 连续添加多个椭球到同一图

ellipsoid(
    a=1,
    b=4,
    c=1,
    center=(0, 0, 0),
    v_range=(np.pi/6, np.pi/2),
    u_range=(-np.pi/3, np.pi/3),
    color="rgba(0, 255, 0, 0.2)",
    name="Green Ellipsoid",
)
ellipsoid(
    a=1,
    b=4,
    c=1,
    center=(0, 0, 0),
    v_range=(-np.pi / 2, 0),
    u_range=(-np.pi / 2, np.pi / 2),
    color="rgba(255, 0, 0, 0.2)",
    name="Blue Ellipsoid",
)

# 显示组合图
show_figure(fig)

In [ ]:
# ========== 高级切片：平面与测地线交集 ==========

def ellipsoid_plane_slice(
    a=1, b=1, c=1,
    center=(0, 0, 0),
    rotation=None,
    plane_normal=(0, 0, 1),
    plane_d=0,
    u_res=40,
    v_res=40,
    color="rgba(100,150,255,0.7)",
    name="Plane Slice",
    fig=None,
    auto_add=True,
    **kwargs,
):
    """
    通过平面切割椭球面，平面方程: n_x*x + n_y*y + n_z*z = d
    
    参数：
        a, b, c: 椭球三个半轴
        center: 椭球中心
        rotation: 旋转角度 (rx, ry, rz)
        plane_normal: 平面法向量 (n_x, n_y, n_z)
        plane_d: 平面方程常数项（相对于椭球中心）
        u_res, v_res: 网格分辨率
        color: 颜色
        name: 图例名称
        fig: Figure 对象
        auto_add: 是否自动添加到全局图
        
    返回：
        fig (go.Figure)
    
    示例：
        # 竖直平面 (x=0)
        fig = ellipsoid_plane_slice(plane_normal=(1,0,0), plane_d=0)
        
        # 倾斜平面 (x+y=0)
        fig = ellipsoid_plane_slice(plane_normal=(1,1,0), plane_d=0)
    """
    global _current_fig
    
    # 归一化法向量
    nx, ny, nz = plane_normal
    norm = np.sqrt(nx**2 + ny**2 + nz**2)
    nx, ny, nz = nx/norm, ny/norm, nz/norm
    
    # 生成参数网格
    u = np.linspace(0, 2*np.pi, u_res)
    v = np.linspace(0, np.pi, v_res)
    u_grid, v_grid = np.meshgrid(u, v)
    
    # 椭球参数方程
    x_param = a * np.cos(u_grid) * np.sin(v_grid)
    y_param = b * np.sin(u_grid) * np.sin(v_grid)
    z_param = c * np.cos(v_grid)
    
    # 应用旋转
    if rotation:
        rx, ry, rz = (
            np.radians(rotation[0]),
            np.radians(rotation[1]),
            np.radians(rotation[2]),
        )
        Rx = np.array([[1, 0, 0], [0, np.cos(rx), -np.sin(rx)], [0, np.sin(rx), np.cos(rx)]])
        Ry = np.array([[np.cos(ry), 0, np.sin(ry)], [0, 1, 0], [-np.sin(ry), 0, np.cos(ry)]])
        Rz = np.array([[np.cos(rz), -np.sin(rz), 0], [np.sin(rz), np.cos(rz), 0], [0, 0, 1]])
        R = Rz @ Ry @ Rx
        
        pts = np.stack([x_param, y_param, z_param], axis=-1)
        shape = pts.shape
        pts_flat = pts.reshape(-1, 3)
        pts_rot = pts_flat @ R.T
        pts_rotated = pts_rot.reshape(shape)
        x_param, y_param, z_param = pts_rotated[..., 0], pts_rotated[..., 1], pts_rotated[..., 2]
    
    # 应用平移
    x_param = x_param + center[0]
    y_param = y_param + center[1]
    z_param = z_param + center[2]
    
    # 计算点到平面的距离（带符号）
    # 平面方程: nx(x-cx) + ny(y-cy) + nz(z-cz) = d
    dist = nx*(x_param - center[0]) + ny*(y_param - center[1]) + nz*(z_param - center[2]) - plane_d
    
    # 只保留平面附近的点（距离较小）
    tolerance = 0.3  # 切片厚度
    mask = np.abs(dist) < tolerance
    
    # 使用掩码提取切片点
    x_slice = np.where(mask, x_param, np.nan)
    y_slice = np.where(mask, y_param, np.nan)
    z_slice = np.where(mask, z_param, np.nan)
    
    # 确定要使用的 Figure
    if fig is None:
        if auto_add and _current_fig is not None:
            fig = _current_fig
        else:
            fig = go.Figure()
    
    surface_kwargs = {k: v for k, v in kwargs.items() if k not in ['u_range', 'v_range']}
    
    fig.add_trace(
        go.Surface(
            x=x_slice,
            y=y_slice,
            z=z_slice,
            surfacecolor=np.ones_like(z_slice),
            colorscale=[[0, color], [1, color]],
            showscale=False,
            name=name,
            **surface_kwargs,
        )
    )
    
    fig.update_layout(
        scene=dict(aspectmode='data'),
        hovermode="closest",
    )
    
    return fig


def ellipsoid_wedge(
    a=1, b=1, c=1,
    center=(0, 0, 0),
    rotation=None,
    plane1_normal=(1, 0, 0),
    plane1_d=0,
    plane2_normal=(0, 1, 0),
    plane2_d=0,
    u_res=40,
    v_res=40,
    color="rgba(100,150,255,0.7)",
    name="Wedge",
    fig=None,
    auto_add=True,
    **kwargs,
):
    """
    通过两个平面切割椭球，获得棱形（楔形）区域。
    保留满足以下条件的点：
        n1·(p-center) ≤ d1  AND  n2·(p-center) ≤ d2
    
    参数：
        plane1_normal, plane1_d: 第一个平面的法向量和常数项
        plane2_normal, plane2_d: 第二个平面的法向量和常数项
        其他参数同 ellipsoid()
    
    返回：
        fig (go.Figure)
    
    示例：
        # 获得 x≥0 且 y≥0 的四分之一区域
        fig = ellipsoid_wedge(
            plane1_normal=(-1,0,0), plane1_d=0,
            plane2_normal=(0,-1,0), plane2_d=0,
            color='red'
        )
        
        # 获得 x+y≤0 且 x-y≤0 的区域
        fig = ellipsoid_wedge(
            plane1_normal=(1,1,0), plane1_d=0,
            plane2_normal=(1,-1,0), plane2_d=0,
            color='blue'
        )
    """
    global _current_fig
    
    # 归一化法向量
    def normalize(v):
        norm = np.sqrt(sum(x**2 for x in v))
        return tuple(x/norm for x in v)
    
    nx1, ny1, nz1 = normalize(plane1_normal)
    nx2, ny2, nz2 = normalize(plane2_normal)
    
    # 生成参数网格
    u = np.linspace(0, 2*np.pi, u_res)
    v = np.linspace(0, np.pi, v_res)
    u_grid, v_grid = np.meshgrid(u, v)
    
    # 椭球参数方程
    x_param = a * np.cos(u_grid) * np.sin(v_grid)
    y_param = b * np.sin(u_grid) * np.sin(v_grid)
    z_param = c * np.cos(v_grid)
    
    # 应用旋转
    if rotation:
        rx, ry, rz = (
            np.radians(rotation[0]),
            np.radians(rotation[1]),
            np.radians(rotation[2]),
        )
        Rx = np.array([[1, 0, 0], [0, np.cos(rx), -np.sin(rx)], [0, np.sin(rx), np.cos(rx)]])
        Ry = np.array([[np.cos(ry), 0, np.sin(ry)], [0, 1, 0], [-np.sin(ry), 0, np.cos(ry)]])
        Rz = np.array([[np.cos(rz), -np.sin(rz), 0], [np.sin(rz), np.cos(rz), 0], [0, 0, 1]])
        R = Rz @ Ry @ Rx
        
        pts = np.stack([x_param, y_param, z_param], axis=-1)
        shape = pts.shape
        pts_flat = pts.reshape(-1, 3)
        pts_rot = pts_flat @ R.T
        pts_rotated = pts_rot.reshape(shape)
        x_param, y_param, z_param = pts_rotated[..., 0], pts_rotated[..., 1], pts_rotated[..., 2]
    
    # 应用平移
    x_param = x_param + center[0]
    y_param = y_param + center[1]
    z_param = z_param + center[2]
    
    # 计算点到两个平面的有向距离
    dist1 = nx1*(x_param - center[0]) + ny1*(y_param - center[1]) + nz1*(z_param - center[2]) - plane1_d
    dist2 = nx2*(x_param - center[0]) + ny2*(y_param - center[1]) + nz2*(z_param - center[2]) - plane2_d
    
    # 保留两个条件都满足的点（两个半平面的交集）
    mask = (dist1 <= 0) & (dist2 <= 0)
    
    x_wedge = np.where(mask, x_param, np.nan)
    y_wedge = np.where(mask, y_param, np.nan)
    z_wedge = np.where(mask, z_param, np.nan)
    
    # 确定要使用的 Figure
    if fig is None:
        if auto_add and _current_fig is not None:
            fig = _current_fig
        else:
            fig = go.Figure()
    
    surface_kwargs = {k: v for k, v in kwargs.items() if k not in ['u_range', 'v_range']}
    
    fig.add_trace(
        go.Surface(
            x=x_wedge,
            y=y_wedge,
            z=z_wedge,
            surfacecolor=np.ones_like(z_wedge),
            colorscale=[[0, color], [1, color]],
            showscale=False,
            name=name,
            **surface_kwargs,
        )
    )
    
    fig.update_layout(
        scene=dict(aspectmode='data'),
        hovermode="closest",
    )
    
    return fig


# ========== 演示：平面切片与楔形区域 ==========
print("演示 1: 单个平面切片 - 竖直平面 (x=0)")
fig_plane1 = ellipsoid_plane_slice(
    a=2, b=1.5, c=1,
    plane_normal=(1, 0, 0),
    plane_d=0,
    color='rgba(255, 0, 0, 0.8)',
    name='Vertical Plane (x=0)'
)
fig_plane1.show()

print("\n演示 2: 单个平面切片 - 倾斜平面 (x+y=0)")
fig_plane2 = ellipsoid_plane_slice(
    a=2, b=1.5, c=1,
    plane_normal=(1, 1, 0),
    plane_d=0,
    color='rgba(0, 255, 0, 0.8)',
    name='Diagonal Plane (x+y=0)'
)
fig_plane2.show()

print("\n演示 3: 楔形区域 - 两条垂直平面的交集 (x≥0 且 y≥0)")
fig_wedge1 = ellipsoid_wedge(
    a=2, b=1.5, c=1,
    plane1_normal=(-1, 0, 0),  # x ≥ 0
    plane1_d=0,
    plane2_normal=(0, -1, 0),  # y ≥ 0
    plane2_d=0,
    color='rgba(0, 0, 255, 0.8)',
    name='Wedge: x≥0, y≥0'
)
fig_wedge1.show()

print("\n演示 4: 楔形区域 - 倾斜平面的交集 (x+y≤0 且 x-y≤0)")
fig_wedge2 = ellipsoid_wedge(
    a=2, b=1.5, c=1,
    plane1_normal=(1, 1, 0),
    plane1_d=0,
    plane2_normal=(1, -1, 0),
    plane2_d=0,
    color='rgba(255, 255, 0, 0.8)',
    name='Wedge: x+y≤0, x-y≤0'
)
fig_wedge2.show()

print("\n演示 5: 组合多个楔形区域 - 创建复杂结构")
fig_complex = start_figure("Complex Ellipsoid Structure", width=1200, height=900)

# 第一象限
ellipsoid_wedge(
    a=2, b=1.5, c=1,
    plane1_normal=(-1, 0, 0), plane1_d=0,
    plane2_normal=(0, -1, 0), plane2_d=0,
    color='rgba(255, 0, 0, 0.7)',
    name='Q1: x≥0, y≥0',
    fig=fig_complex
)

# 第二象限
ellipsoid_wedge(
    a=2, b=1.5, c=1,
    plane1_normal=(1, 0, 0), plane1_d=0,
    plane2_normal=(0, -1, 0), plane2_d=0,
    color='rgba(0, 255, 0, 0.7)',
    name='Q2: x≤0, y≥0',
    fig=fig_complex
)

# 第三象限
ellipsoid_wedge(
    a=2, b=1.5, c=1,
    plane1_normal=(1, 0, 0), plane1_d=0,
    plane2_normal=(0, 1, 0), plane2_d=0,
    color='rgba(0, 0, 255, 0.7)',
    name='Q3: x≤0, y≤0',
    fig=fig_complex
)

# 第四象限
ellipsoid_wedge(
    a=2, b=1.5, c=1,
    plane1_normal=(-1, 0, 0), plane1_d=0,
    plane2_normal=(0, 1, 0), plane2_d=0,
    color='rgba(255, 255, 0, 0.7)',
    name='Q4: x≥0, y≤0',
    fig=fig_complex
)

show_figure(fig_complex)

In [ ]:
fig_complex = start_figure("Complex Ellipsoid Structure", width=1200, height=900)

# 第一象限
ellipsoid_wedge(
    a=4,
    b=1,
    c=1,
    plane1_normal=(-1, 0, 0),
    plane1_d=0,
    plane2_normal=(0, -1, 0),
    plane2_d=0,
    color="rgba(255, 0, 0, 0.7)",
    name="Q1: x≥0, y≥0",
    fig=fig_complex,
)
show_figure(fig_complex)

In [47]:
# ========== 精确楔形区域（解析边界，光滑区域） ==========

def plane_ellipsoid_intersection(a, b, c, center, plane_normal, plane_d, n_points=800):
    """
    解析求平面与椭球的交线（若存在），返回 (x,y,z) arrays 或 None。
    平面形式： n·(p - center) = plane_d
    """
    n = np.array(plane_normal, dtype=float)
    n = n / np.linalg.norm(n)
    p0 = np.array(center, dtype=float) + n * plane_d

    # basis on plane
    arbitrary = np.array([1.0, 0.0, 0.0])
    if abs(np.dot(arbitrary, n)) > 0.9:
        arbitrary = np.array([0.0, 1.0, 0.0])
    e1 = np.cross(n, arbitrary)
    e1 /= np.linalg.norm(e1)
    e2 = np.cross(n, e1)
    e2 /= np.linalg.norm(e2)

    S = np.diag([1.0 / a ** 2, 1.0 / b ** 2, 1.0 / c ** 2])
    E = np.column_stack([e1, e2])
    M = E.T.dot(S).dot(E)
    l = E.T.dot(S).dot(p0)
    s = p0.dot(S).dot(p0) - 1.0

    # invert M
    try:
        Minv = np.linalg.inv(M)
    except np.linalg.LinAlgError:
        return None

    u0 = -Minv.dot(l)
    cprime = u0.dot(M).dot(u0) - s
    if cprime <= 0:
        return None

    vals, vecs = np.linalg.eigh(M)
    if np.any(vals <= 0):
        return None
    axes_lengths = np.sqrt(cprime / vals)

    t = np.linspace(0, 2 * np.pi, n_points)
    circle = np.vstack([np.cos(t), np.sin(t)])
    uv_local = (vecs @ (axes_lengths[:, None] * circle)) + u0[:, None]
    pts3 = p0[:, None] + E.dot(uv_local)
    x, y, z = pts3[0, :], pts3[1, :], pts3[2, :]
    return x, y, z


def ellipsoid_wedge_exact(
    a=1, b=1, c=1,
    center=(0, 0, 0),
    rotation=None,
    plane1_normal=(1,0,0), plane1_d=0,
    plane2_normal=(0,1,0), plane2_d=0,
    u_res=200,
    v_res=200,
    below_xy=False,
    color="rgba(100,150,255,0.8)",
    name="WedgeExact",
    fig=None,
):
    """
    用解析边界构建楔形区域。此实现避免为每个窄带添加单独的 Surface，
    而是在整个 (u,v) 网格上生成一次带掩码的数据并绘制单个 Surface，从而避免
    产生大量 trace 导致的崩溃。

    当 below_xy=True 时只保留 z < 0 的部分（XY 平面下）。
    """
    global _current_fig
    if fig is None:
        fig = _current_fig if _current_fig is not None else go.Figure()

    # normalize normals
    n1 = np.array(plane1_normal, dtype=float); n1 /= np.linalg.norm(n1)
    n2 = np.array(plane2_normal, dtype=float); n2 /= np.linalg.norm(n2)

    # construct full param grid
    us = np.linspace(0.0, 2 * np.pi, u_res)
    vs = np.linspace(0.0, np.pi, v_res)
    u_grid, v_grid = np.meshgrid(us, vs)

    # compute points
    X = a * np.cos(u_grid) * np.sin(v_grid) + center[0]
    Y = b * np.sin(u_grid) * np.sin(v_grid) + center[1]
    Z = c * np.cos(v_grid) + center[2]

    # compute plane inequalities
    # vectorized dot product: (3,n,m)·n -> (n,m)
    # compute vectors from center
    VX = X - center[0]
    VY = Y - center[1]
    VZ = Z - center[2]

    cond1 = (n1[0]*VX + n1[1]*VY + n1[2]*VZ) <= (plane1_d + 1e-9)
    cond2 = (n2[0]*VX + n2[1]*VY + n2[2]*VZ) <= (plane2_d + 1e-9)
    cond3 = (Z < 0) if below_xy else np.ones_like(Z, dtype=bool)

    mask = cond1 & cond2 & cond3

    # create masked arrays with NaN outside region
    Xm = np.where(mask, X, np.nan)
    Ym = np.where(mask, Y, np.nan)
    Zm = np.where(mask, Z, np.nan)

    # Add single Surface trace (masked points become holes)
    fig.add_trace(
        go.Surface(
            x=Xm,
            y=Ym,
            z=Zm,
            surfacecolor=np.ones_like(Zm),
            colorscale=[[0, color], [1, color]],
            showscale=False,
            name=name,
            hoverinfo='skip',
        )
    )

    # 增加两条解析边界的高分辨率曲线并裁剪到区域
    res_edge = max(400, u_res * 2)
    eb1 = plane_ellipsoid_intersection(a,b,c,center,n1,plane1_d,n_points=res_edge)
    eb2 = plane_ellipsoid_intersection(a,b,c,center,n2,plane2_d,n_points=res_edge)
    if eb1 is not None:
        xb1,yb1,zb1 = eb1
        # mask by other plane and below_xy
        mask_edge = (n2[0]*(xb1-center[0]) + n2[1]*(yb1-center[1]) + n2[2]*(zb1-center[2])) <= (plane2_d + 1e-9)
        if below_xy:
            mask_edge &= (zb1 < 0)
        fig.add_trace(
            go.Scatter3d(
                x=xb1[mask_edge], y=yb1[mask_edge], z=zb1[mask_edge],
                mode='lines', line=dict(color=color.replace('0.8','1.0') if isinstance(color,str) else color, width=6),
                name=name + '_edge1'
            )
        )
    if eb2 is not None:
        xb2,yb2,zb2 = eb2
        mask_edge = (n1[0]*(xb2-center[0]) + n1[1]*(yb2-center[1]) + n1[2]*(zb2-center[2])) <= (plane1_d + 1e-9)
        if below_xy:
            mask_edge &= (zb2 < 0)
        fig.add_trace(
            go.Scatter3d(
                x=xb2[mask_edge], y=yb2[mask_edge], z=zb2[mask_edge],
                mode='lines', line=dict(color=color.replace('0.8','1.0') if isinstance(color,str) else color, width=6),
                name=name + '_edge2'
            )
        )

    fig.update_layout(scene=dict(aspectmode='data'))
    return fig


# ========== 在当前 notebook 中演示（针对用户参数） ==========
print('绘制解析楔形区域（z<0）示例： a=4,b=1,c=1，过焦点，和 Y 轴向下夹角30°的两平面')

# 参数
A, B, C = 4, 1, 1
c_dist = np.sqrt(A**2 - B**2)
focal_pt = (c_dist, 0, 0)
angle = np.radians(30)
plane_ny = np.cos(angle)
plane_nz = np.sin(angle)
plane1_n = (0, plane_ny, -plane_nz)
plane2_n = (0, plane_ny, plane_nz)

fig_exact = start_figure('Exact Focal Wedge (z<0)', width=1000, height=800)
# 参考半透明椭球
ellipsoid(a=A, b=B, c=C, center=(0,0,0), color='rgba(200,200,200,0.07)', u_res=80, v_res=80, fig=fig_exact)
# 精确楔形（仅 z<0）
ellipsoid_wedge_exact(
    a=A, b=B, c=C, center=(0,0,0),
    plane1_normal=plane1_n, plane1_d=0,
    plane2_normal=plane2_n, plane2_d=0,
    u_res=300, v_res=200, below_xy=True,
    color='rgba(255,100,100,0.8)', name='ExactWedge', fig=fig_exact
)
# 标注焦点
fig_exact.add_trace(go.Scatter3d(x=[focal_pt[0]], y=[0], z=[0], mode='markers+text', marker=dict(size=6,color='red'), text=['F'], textposition='top center'))
show_figure(fig_exact)
print('完成：已在 notebook 中绘制解析楔形区域（z<0）。')


绘制解析楔形区域（z<0）示例： a=4,b=1,c=1，过焦点，和 Y 轴向下夹角30°的两平面


完成：已在 notebook 中绘制解析楔形区域（z<0）。


In [ ]:
# ========== 通过焦点的两平面切割 ==========
import math

# 椭球参数
a, b, c = 4, 1, 1

# 计算焦点（椭长轴方向）
c_dist = math.sqrt(a**2 - b**2)  # sqrt(16 - 1) = sqrt(15) ≈ 3.873
focal_point = (c_dist, 0, 0)  # 取右焦点

print(f"椭球参数: a={a}, b={b}, c={c}")
print(f"焦距参数 c_dist = sqrt({a}² - {b}²) = sqrt({a**2 - b**2}) = {c_dist:.4f}")
print(f"焦点: ({c_dist:.4f}, 0, 0)")

# 两个平面的定义
# 平面都过焦点，平行于X轴，与Y轴向下夹角30°
# 法向量在yz平面上，与y轴夹角30°

# 平面1: 与Y轴夹角+30°（向下）→ 法向量 (0, cos(60°), -sin(60°)) = (0, 0.5, -√3/2)
# 平面2: 与Y轴夹角-30°（向下）→ 法向量 (0, cos(60°), sin(60°)) = (0, 0.5, √3/2)

angle_deg = 30
angle_rad = math.radians(angle_deg)

# 夹角30°对应的法向量分量关系：
# cos(angle) = |n_y| / sqrt(n_y² + n_z²)
# sin(angle) = |n_z| / sqrt(n_y² + n_z²)

n_y_norm = math.cos(angle_rad)  # cos(30°) ≈ 0.866
n_z_norm = math.sin(angle_rad)  # sin(30°) = 0.5

# 两个平面的法向量（向下夹角，z分量为负）
plane1_normal = (0, n_y_norm, -n_z_norm)    # 平面1: 向Y+Z向下倾斜
plane2_normal = (0, n_y_norm, n_z_norm)     # 平面2: 向Y-Z向下倾斜

# 平面过焦点，方程为: n_y * (y - 0) + n_z * (z - 0) = 0
# 即: n_y * y + n_z * z = 0
plane_d = 0

print(f"\n平面1法向量: {plane1_normal}")
print(f"平面2法向量: {plane2_normal}")
print(f"两平面过焦点 ({focal_point[0]:.4f}, 0, 0)")

# 绘制两平面的交集（楔形区域）- 这是较小的部分
fig_focal = start_figure("通过焦点的两平面切割", width=1000, height=800)

# 添加完整椭球作为参考（半透明）
ellipsoid(
    a=a, b=b, c=c,
    center=(0, 0, 0),
    color='rgba(200, 200, 200, 0.1)',
    name='Ellipsoid (reference)',
    u_res=50, v_res=50,
    fig=fig_focal
)

# 添加两平面的交集（楔形，较小部分）
ellipsoid_wedge(
    a=a, b=b, c=c,
    center=(0, 0, 0),
    plane1_normal=plane1_normal,
    plane1_d=0,
    plane2_normal=plane2_normal,
    plane2_d=0,
    u_res=50, v_res=50,
    color='rgba(255, 100, 100, 0.8)',
    name='Focal Wedge (smaller part)',
    fig=fig_focal
)

# 添加焦点作为标记
fig_focal.add_trace(
    go.Scatter3d(
        x=[focal_point[0]],
        y=[focal_point[1]],
        z=[focal_point[2]],
        mode='markers',
        marker=dict(size=8, color='red', symbol='diamond'),
        name='Focal Point',
        showlegend=True
    )
)

# 添加另一个焦点
fig_focal.add_trace(
    go.Scatter3d(
        x=[-focal_point[0]],
        y=[focal_point[1]],
        z=[focal_point[2]],
        mode='markers',
        marker=dict(size=8, color='blue', symbol='diamond'),
        name='Other Focal Point',
        showlegend=True
    )
)

show_figure(fig_focal)

print(f"\n✓ 绘制完成：通过焦点的两平面切割显示较小的楔形区域")

# ========== XY平面下的切片 ==========
print("\n" + "="*60)
print("XY平面下的投影分析")
print("="*60)

# 在XY平面上投影：只看椭圆 x²/a² + y²/b² = 1（当 z=0）
fig_xy = go.Figure()

# 绘制椭圆
theta = np.linspace(0, 2*np.pi, 1000)
ellipse_x = a * np.cos(theta)
ellipse_y = b * np.sin(theta)

fig_xy.add_trace(
    go.Scatter(
        x=ellipse_x,
        y=ellipse_y,
        mode='lines',
        line=dict(color='blue', width=2),
        name='Ellipse (z=0)',
        fill='toself',
        fillcolor='rgba(0, 0, 255, 0.1)'
    )
)

# 在XY平面上绘制两条直线（平面与XY平面z=0的交线）
# 平面方程: n_y * y + n_z * z = 0
# z=0时: n_y * y = 0 → y = 0（X轴）

# 两条平面在z=0处都交于X轴（y=0的直线）
fig_xy.add_trace(
    go.Scatter(
        x=ellipse_x,
        y=[0]*len(ellipse_x),
        mode='lines',
        line=dict(color='red', width=3, dash='dash'),
        name='Plane intersections (z=0)'
    )
)

# 添加焦点标记
fig_xy.add_trace(
    go.Scatter(
        x=[focal_point[0], -focal_point[0]],
        y=[0, 0],
        mode='markers+text',
        marker=dict(size=10, color=['red', 'blue']),
        text=[f'F₁({focal_point[0]:.2f},0)', f'F₂({-focal_point[0]:.2f},0)'],
        textposition='top center',
        name='Focal Points'
    )
)

fig_xy.update_layout(
    title='XY平面投影 (z=0)',
    xaxis_title='X (a轴)',
    yaxis_title='Y (b轴)',
    width=800,
    height=700,
    showlegend=True,
    hovermode='closest',
    xaxis=dict(scaleanchor='y', scaleratio=1),
    yaxis=dict(scaleanchor='x', scaleratio=1)
)

fig_xy.show()

print(f"\n注：在XY平面(z=0)上，两条平面都与X轴重合（y=0）")
print(f"    较小的切片在XY平面下被投影为X轴的一段")


In [ ]:
import numpy as np


def plane_ellipsoid_intersection(a, b, c, center, plane_normal, plane_d, n_points=800):
    """
    解析求平面与椭球的交线（若有），返回 (x,y,z) arrays (n_points,)。
    椭球中心 = center，椭球方程标准形式 (x/a)^2+(y/b)^2+(z/c)^2=1（相对于 center）。
    平面方程 (n · (p - center)) = plane_d  （plane_d 相对于 center 的距离）
    plane_normal 不必归一化。
    返回 None 当无交（空集）或者退化时。
    """
    # normalize plane normal
    n = np.array(plane_normal, dtype=float)
    n = n / np.linalg.norm(n)

    # point on plane p0: choose center + n * plane_d
    p0 = np.array(center, dtype=float) + n * plane_d

    # build orthonormal basis (e1,e2) spanning plane
    # pick arbitrary vector not parallel to n
    arbitrary = np.array([1.0, 0.0, 0.0])
    if abs(np.dot(arbitrary, n)) > 0.9:
        arbitrary = np.array([0.0, 1.0, 0.0])
    e1 = np.cross(n, arbitrary)
    e1 /= np.linalg.norm(e1)
    e2 = np.cross(n, e1)
    e2 /= np.linalg.norm(e2)

    # ellipsoid quadratic form S = diag(1/a^2,1/b^2,1/c^2)
    S = np.diag([1.0 / a**2, 1.0 / b**2, 1.0 / c**2])

    # compute 2x2 matrix M, vector l, scalar s for quadratic in (u,v):
    # (p0 + u e1 + v e2)^T S (p0 + u e1 + v e2) = 1
    # => [u v] M [u;v] + 2 l^T [u;v] + s = 0
    E = np.column_stack([e1, e2])  # 3x2
    M = E.T.dot(S).dot(E)  # 2x2
    l = E.T.dot(S).dot(p0)  # 2
    s = p0.dot(S).dot(p0) - 1.0  # scalar

    # check if M is positive definite; if not, no proper ellipse
    # solve for center u0 = - M^{-1} l
    try:
        Minv = np.linalg.inv(M)
    except np.linalg.LinAlgError:
        return None

    u0 = -Minv.dot(l)
    cprime = u0.dot(M).dot(u0) - s
    if cprime <= 0:
        return None  # no real ellipse

    # eigen-decompose M to get principal axes
    vals, vecs = np.linalg.eigh(M)  # vals >0 for ellipse
    if np.any(vals <= 0):
        return None

    axes_lengths = np.sqrt(cprime / vals)  # semi-axes lengths in u/v coordinates

    # param t
    t = np.linspace(0, 2 * np.pi, n_points)
    circle = np.vstack([np.cos(t), np.sin(t)])  # 2 x n
    uv_local = (vecs @ (axes_lengths[:, None] * circle)) + u0[:, None]  # 2 x n

    # map back to 3D
    pts3 = p0[:, None] + E.dot(uv_local)  # 3 x n
    x, y, z = pts3[0, :], pts3[1, :], pts3[2, :]
    return x, y, z


# 假设 fig 是当前3D图，且你已经用 ellipsoid_wedge 绘制了内部面
res = 1000
x_edge, y_edge, z_edge = plane_ellipsoid_intersection(
    a=4, b=1, c=1, center=(0, 0, 0), plane_normal=plane1_normal, plane_d=0, n_points=res
)
fig.add_trace(
    go.Scatter3d(
        x=x_edge,
        y=y_edge,
        z=z_edge,
        mode="lines",
        line=dict(width=6, color="rgba(255,100,100,1)"),
        name="boundary1",
    )
)
# 同理绘制第二条交线
# x2, y2, z2 = plane_ellipsoid_intersection(
#     a=4, b=1, c=1, center=(0, 0, 0), plane_normal=plane1_normal, plane_d=0, n_points=res
# )
# fig.add_trace(
#     go.Scatter3d(
#         x=x2, y=y2, z=z2, mode="lines", line=dict(width=6, color="rgba(255,100,100,1)")
#     )
# )